# Chapter 18 — Simple and Multiple Regression

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch18_regression_mincer.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 18.1 The data: salaried workers in PLFS 2023–24

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

from indiadata import read_plfs

p = read_plfs("perv1", columns=["sex", "age", "gedu_lvl", "form_edu",
                                "acws", "ern_reg"])
s = p[(p["acws"] == 31) & (p["ern_reg"] > 0)
      & p["age"].between(15, 59) & p["sex"].isin([1, 2])]
del p                                            # free the big frame

d = pd.DataFrame({
    "earn": s["ern_reg"].astype(float),          # ₹ per month
    "educ": s["form_edu"].astype(float),         # years of schooling
    "age": s["age"].astype(float),
    "female": (s["sex"] == 2).astype(float),
    "urban": (s["sec"] == 2).astype(float),
    "gedu": s["gedu_lvl"].astype(int),
    "st": s["st"].astype(int),
    "mfsu": s["mfsu"].astype(int),
    "weight": s["weight"].astype(float),
})
del s
d["lwage"] = np.log(d["earn"])
d["exper"] = (d["age"] - d["educ"] - 6).clip(lower=0)
print(f"{len(d):,} salaried workers")
d[["earn", "lwage", "educ", "exper", "female", "urban"]].describe().round(2).T

## 18.2 Look before you regress

In [ ]:
by_years = d.groupby("educ")["lwage"].agg(["mean", "size"])
by_years = by_years[by_years["size"] >= 50]          # drop very thin cells

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.scatter(by_years.index, by_years["mean"], s=by_years["size"] / 25,
           color="#1f4e79", alpha=0.8)
ax.set_xlabel("Years of formal education")
ax.set_ylabel("Mean log monthly earnings")
ax.set_title("Log earnings rise with schooling, faster after year 12",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.05, "Regular wage/salaried workers aged 15–59 (CWS); dot area "
         "proportional to the number of workers; cells with fewer than 50 workers "
         "omitted.\nSource: author's calculation from PLFS 2023–24 unit-level "
         "data (NSO, MoSPI), unweighted.", fontsize=8, color="grey")
plt.show()

## 18.3 Simple regression, three ways

In [ ]:
x, y = d["educ"].to_numpy(), d["lwage"].to_numpy()
b1 = ((x - x.mean()) * (y - y.mean())).sum() / ((x - x.mean()) ** 2).sum()
b0 = y.mean() - b1 * x.mean()
print(f"b0 = {b0:.4f}, b1 = {b1:.4f}")

In [ ]:
X = np.column_stack([np.ones(len(x)), x])
beta = np.linalg.solve(X.T @ X, X.T @ y)
resid = y - X @ beta
n, k = X.shape
sigma2 = resid @ resid / (n - k)                         # error variance
se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))   # classical SEs
r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
print("coefficients:", beta.round(4), " SEs:", se.round(5), f" R² = {r2:.4f}")

In [ ]:
m1 = smf.ols("lwage ~ educ", data=d).fit()
print(m1.summary().tables[1])
print(f"R² = {m1.rsquared:.4f},  n = {int(m1.nobs):,}")

In [ ]:
print(f"One more year of schooling: +{100 * m1.params['educ']:.1f}% (approx.), "
      f"+{100 * np.expm1(m1.params['educ']):.1f}% (exact)")

## 18.4 Multiple regression: the Mincer equation

In [ ]:
mincer = "lwage ~ educ + exper + I(exper**2) + female + urban"
m2 = smf.ols(mincer, data=d).fit()
print(m2.summary().tables[1])
print(f"R² = {m2.rsquared:.3f},  adjusted R² = {m2.rsquared_adj:.3f}")

In [ ]:
print(f"Correlation between schooling and experience: "
      f"{d['educ'].corr(d['exper']):.2f}")

In [ ]:
e_educ = smf.ols("educ ~ exper + I(exper**2) + female + urban", data=d).fit().resid
fwl = sm.OLS(d["lwage"], sm.add_constant(e_educ.rename("educ_resid"))).fit()
print(f"Multiple regression: {m2.params['educ']:.6f}   "
      f"FWL two-step: {fwl.params['educ_resid']:.6f}")

## 18.5 Testing hypotheses: $t$ and $F$

In [ ]:
print(m2.t_test("educ = 0.10"))

In [ ]:
m_r = smf.ols("lwage ~ educ + female + urban", data=d).fit()
q, df_resid = 2, m2.df_resid
F_hand = ((m2.rsquared - m_r.rsquared) / q) / ((1 - m2.rsquared) / df_resid)
print(f"F by hand = {F_hand:.1f}")
print(m2.f_test("exper = 0, I(exper ** 2) = 0"))

## 18.6 Dummy variables: education levels instead of years

In [ ]:
LEVEL = {1: "No formal schooling", 2: "No formal schooling",
         3: "No formal schooling", 4: "No formal schooling",
         5: "Below primary", 6: "Primary", 7: "Middle", 8: "Secondary",
         10: "Higher secondary", 11: "Diploma/certificate", 12: "Graduate",
         13: "Postgraduate+"}
ORDER = list(dict.fromkeys(LEVEL.values()))
d["edlevel"] = pd.Categorical(d["gedu"].map(LEVEL), categories=ORDER)
d.groupby("edlevel", observed=True).agg(workers=("educ", "size"),
                                        mean_years=("educ", "mean")).round(1).T

In [ ]:
m3 = smf.ols("lwage ~ C(edlevel) + exper + I(exper**2) + female + urban",
             data=d).fit()
lev = (m3.params.filter(like="edlevel")
       .rename(lambda s: s.split("T.")[1].rstrip("]")))
ci = (m3.conf_int().filter(like="edlevel", axis=0)
      .rename(lambda s: s.split("T.")[1].rstrip("]")))
pd.DataFrame({"coef": lev, "% vs no schooling": 100 * np.expm1(lev)}).round(2)

In [ ]:
yrs = d.groupby("edlevel", observed=True)["educ"].mean()
effect = pd.concat([pd.Series({ORDER[0]: 0.0}), lev])     # base level = 0
ladder = [l for l in ORDER if l != "Diploma/certificate"]
per_year = (effect[ladder].diff() / yrs[ladder].diff()).iloc[1:]
(100 * per_year).round(1).rename("log points per extra year").to_frame().T

In [ ]:
w_lv = d["edlevel"].value_counts().reindex(ORDER)
shift = np.average(effect - m2.params["educ"] * yrs, weights=w_lv)
implied = shift + m2.params["educ"] * yrs

fig, ax = plt.subplots(figsize=(8.5, 4.4))
ax.errorbar(yrs.iloc[1:], lev, yerr=[lev - ci[0], ci[1] - lev], fmt="o",
            color="#1f4e79", capsize=3, label="Education-level dummies (95% CI)")
ax.plot(yrs, implied, color="#c55a11", lw=1.5,
        label=f"Linear in years (slope {m2.params['educ']:.2f})")
ax.plot(0, 0, "o", color="#1f4e79", mfc="white")             # the base level
ax.annotate("No formal schooling (base)", (0, 0), textcoords="offset points",
            xytext=(6, 6), fontsize=8)
for name, xv, yv in zip(lev.index, yrs.iloc[1:], lev):
    ax.annotate(name, (xv, yv), textcoords="offset points", xytext=(-8, 7),
                fontsize=8, ha="right")
ax.set_xlabel("Average years of schooling at each level")
ax.set_ylabel("Log earnings relative to no schooling")
ax.set_xlim(-0.5, 19)
ax.set_title("Returns to education are convex: college pays most",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower right")
fig.text(0.01, -0.05, "OLS with experience, experience², female and urban "
         "controls. Source: author's calculation from PLFS 2023–24 unit-level "
         "data (NSO, MoSPI), unweighted.", fontsize=8, color="grey")
plt.show()

## 18.7 Interactions: is the return the same for women and men?

In [ ]:
m4 = smf.ols("lwage ~ educ * female + exper + I(exper**2) + urban", data=d).fit()
print(m4.summary().tables[1])
ret_m = m4.params["educ"]
ret_f = m4.params["educ"] + m4.params["educ:female"]
print(f"Return to a year of schooling: men {100 * ret_m:.1f}%, "
      f"women {100 * ret_f:.1f}% (approx.)")

In [ ]:
for yrs_ in [0, 10, 15]:
    g = m4.params["female"] + yrs_ * m4.params["educ:female"]
    print(f"{yrs_:>2} years of schooling: gender gap {g:.3f} log points "
          f"({100 * np.expm1(g):.0f}%)")

## 18.8 The age–earnings profile

In [ ]:
m5 = smf.ols("lwage ~ educ + (exper + I(exper**2)) * female + urban",
             data=d).fit()
b = m5.params
for sex, f in [("men", 0), ("women", 1)]:
    b2 = b["exper"] + f * b["exper:female"]
    b3 = b["I(exper ** 2)"] + f * b["I(exper ** 2):female"]
    print(f"{sex:>5}: first year of experience +{100 * b2:.1f}%, "
          f"at 20 years +{100 * (b2 + 2 * b3 * 20):.1f}% a year, "
          f"peak at {-b2 / (2 * b3):.0f} years")

In [ ]:
grid = pd.DataFrame({"exper": np.arange(0, 41), "educ": 15.0, "urban": 1.0})

fig, ax = plt.subplots(figsize=(8, 4.2))
for f, colour, label in [(0, "#1f4e79", "Men"), (1, "#c55a11", "Women")]:
    pred = m5.predict(grid.assign(female=float(f)))
    ax.plot(grid["exper"], np.exp(pred) / 1000, color=colour, lw=2, label=label)
ax.set_xlabel("Years of potential experience")
ax.set_ylabel("Predicted monthly earnings, ₹ thousand")
ax.set_title("Age–earnings profiles of urban graduates", loc="left", fontsize=13)
ax.set_ylim(bottom=0)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
fig.text(0.01, -0.05, "Predictions exp(fitted log earnings) from a Mincer "
         "regression with sex-specific experience terms; 15 years of schooling, "
         "urban.\nSource: author's calculation from PLFS 2023–24 unit-level "
         "data (NSO, MoSPI), unweighted.", fontsize=8, color="grey")
plt.show()

## 18.9 Prediction: a mean and an individual

In [ ]:
new = pd.DataFrame({"educ": [15.0], "exper": [10.0], "female": [1.0],
                    "urban": [1.0]})
pr = m2.get_prediction(new).summary_frame(alpha=0.05)
in_rupees = np.exp(pr[["mean", "mean_ci_lower", "mean_ci_upper",
                       "obs_ci_lower", "obs_ci_upper"]]).round(-2)
in_rupees

In [ ]:
smear = np.exp(m2.resid).mean()
print(f"Smearing factor {smear:.3f}: predicted mean "
      f"₹{np.exp(pr['mean'].iloc[0]) * smear:,.0f} vs "
      f"exp(fitted) ₹{np.exp(pr['mean'].iloc[0]):,.0f}")

## 18.10 State fixed effects

In [ ]:
m6 = smf.ols(mincer + " + C(st)", data=d).fit()
print(f"educ: {m2.params['educ']:.4f} without, {m6.params['educ']:.4f} with "
      f"state effects")
print(f"female: {m2.params['female']:.4f} without, {m6.params['female']:.4f} with")
print(f"R²: {m2.rsquared:.3f} → {m6.rsquared:.3f}")
state_terms = [t for t in m6.params.index if t.startswith("C(st)")]
ft = m6.f_test(", ".join(f"{t} = 0" for t in state_terms))
print(f"F test that all {len(state_terms)} state effects are zero: "
      f"F = {float(np.squeeze(ft.fvalue)):.1f}, p = {float(ft.pvalue):.3g}")

## 18.11 To weight or not to weight?

In [ ]:
m2w = smf.wls(mincer, data=d, weights=d["weight"]).fit()
compare = pd.DataFrame({"OLS (unweighted)": m2.params,
                        "WLS (survey weights)": m2w.params}).round(4)
compare

In [ ]:
m7 = smf.ols("lwage ~ educ * urban + exper + I(exper**2) + female", data=d).fit()
print(f"Return to schooling: rural {100 * m7.params['educ']:.1f}%, "
      f"urban {100 * (m7.params['educ'] + m7.params['educ:urban']):.1f}% "
      f"(interaction t = {m7.tvalues['educ:urban']:.1f})")

## 18.12 Decomposing the gender gap: Oaxaca–Blinder

In [ ]:
Xall = pd.concat([d[["educ", "exper"]].assign(exper2=d["exper"] ** 2),
                  d[["urban"]],
                  pd.get_dummies(d["st"], prefix="st", drop_first=True,
                                 dtype=float)], axis=1)
Xall = sm.add_constant(Xall)
men, wom = d["female"] == 0, d["female"] == 1

fit_m = sm.OLS(d.loc[men, "lwage"], Xall[men]).fit()
fit_f = sm.OLS(d.loc[wom, "lwage"], Xall[wom]).fit()
xbar_m, xbar_f = Xall[men].mean(), Xall[wom].mean()

gap = d.loc[men, "lwage"].mean() - d.loc[wom, "lwage"].mean()
explained = (xbar_m - xbar_f) * fit_m.params           # one term per regressor
unexplained = xbar_f * (fit_m.params - fit_f.params)
print(f"Raw gap: {gap:.3f} log points ({100 * np.expm1(gap):.0f}% higher for men)")
print(f"Explained:   {explained.sum():.3f}")
print(f"Unexplained: {unexplained.sum():.3f}")
print(f"Check: explained + unexplained = {explained.sum() + unexplained.sum():.3f}")

In [ ]:
d.groupby("female")[["educ", "exper", "urban"]].mean().rename(
    index={0.0: "men", 1.0: "women"}).round(2)

In [ ]:
def group(term):
    if term.startswith("st_"):
        return "State"
    if term.startswith("exper"):
        return "Experience"
    return {"educ": "Schooling", "urban": "Urban", "const": "const"}[term]


parts = explained.groupby(group).sum().drop("const").sort_values()
parts

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
bars = pd.concat([parts, pd.Series({"Unexplained": unexplained.sum()})])
colours = ["#9db7d5"] * len(parts) + ["#c55a11"]
ax.barh(bars.index, bars.values, color=colours)
ax.axvline(0, color="black", lw=0.8)
for i, v in enumerate(bars.values):
    ax.text(v + (0.005 if v >= 0 else -0.005), i, f"{v:+.3f}", va="center",
            ha="left" if v >= 0 else "right", fontsize=9)
ax.set_xlabel("Contribution to the male–female gap in mean log earnings")
ax.set_xlim(min(bars.min(), 0) - 0.08, bars.max() + 0.08)
ax.set_title(f"Oaxaca–Blinder decomposition of a {gap:.2f} log-point gap",
             loc="left", fontsize=13)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.08, "Men's coefficients as the reference. Positive: favours men. "
         "Source: author's calculation from PLFS 2023–24 unit-level data "
         "(NSO, MoSPI), unweighted.", fontsize=8, color="grey")
plt.show()

## 18.13 Is this the *causal* return to education?

## Key ideas

## Exercises

## Data citation and terms of use

## Further reading